# InSb quantum dot embedded in an InAs matrix — a first 3D sketch

A standalone (no dependency on the `aestimo` package) 3D single-band effective-mass
Schrödinger solver for a spherical InSb quantum dot embedded in bulk InAs, including strain
via the analytic Eshelby inclusion solution. This is a deliberately scoped-down **first
version** meant to demonstrate the numerical machinery (3D grid, position-dependent mass,
sparse eigensolver, strain-shifted band edges) rather than a publication-grade calculation.
See the accompanying literature citations for where these methods come from.

## Scope and simplifications (read this before trusting the numbers)

- **Single parabolic band** for each carrier (a scalar conduction-band electron mass, and a
  single heavy-hole-like valence mass for holes) — no multiband k·p mixing. This is the same
  level of approximation as aestimo's own conduction-band treatment, just extended to 3D.
- **Spherical dot**, not the flatter lens/disk shapes real self-assembled dots usually form.
  A sphere is the case with a simple closed-form Eshelby solution; real shapes need either the
  general ellipsoidal Eshelby tensor (elliptic integrals) or a numerical elasticity solve.
- **Homogeneous inclusion approximation**: the Eshelby strain solution used here assumes the
  dot and matrix share the same (matrix) elastic constants — only the *eigenstrain* (lattice
  mismatch) differs. InAs and InSb's elastic constants are reasonably close (~20% apart), so
  this is a defensible leading-order approximation, not an exact treatment.
- **Isotropic elasticity**: real zincblende crystals are elastically anisotropic (3 independent
  constants C11, C12, C44); an isotropic Poisson ratio is estimated here via a Voigt average.
- **Hydrostatic-only band coupling**: only the trace of the strain tensor is used (via the
  conduction/valence deformation potentials a_c, a_v). This happens to be exact for a purely
  dilatational eigenstrain in a sphere (no shear strain arises by symmetry), so this isn't an
  extra approximation on top of the sphere choice — but it does mean there's no valence-band
  heavy/light-hole mixing from shear strain, which a real (non-spherical) dot would have.
- **~6.5% lattice mismatch is large** — harmonic (linear) elasticity is being pushed outside
  its most comfortable regime, and a real InSb/InAs dot this size would very likely partially
  plastically relax (misfit dislocations) rather than stay fully coherent/strained.
- The emission energy computed at the end is the **naive single-particle transition energy**
  (electron level minus hole level). It ignores electron-hole Coulomb attraction, which in a
  type-II system (confirmed below) can be a significant correction — treat it as a rough,
  order-of-magnitude starting point, not a quantitative prediction.

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
%matplotlib inline

HBAR2_OVER_2M0 = 0.0380998  # eV * nm^2 -- standard constant hbar^2/(2*m0), converted from 3.80998 eV*Angstrom^2

## Material parameters (Vurgaftman, Meyer & Ram-Mohan, J. Appl. Phys. 89, 5815 (2001))

These are the same source/values used for the InAs/InSb `aestimo` 1D notebooks (kept in this
same repo's `aestimo_database.py` for reference), restated here directly in eV / Angstrom / m0
units rather than imported, since aestimo's own `database.py` stores elastic constants in a
non-obvious scaled unit specific to its own array-building code.

The unstrained conduction/valence band edges (`fi_e`, `fi_h`) are recomputed here with the
exact same `Band_offset`/`Eg` values and formula used in the companion 1D
`sample_1qw_inas_insb_inas.ipynb` notebook, so the two are on the same absolute energy scale
and can be sanity-checked against each other.

In [ ]:
materials = {
    'InAs': dict(
        m_e=0.026, m_h=0.41,           # conduction / heavy-hole effective mass (m0)
        a0=6.0583,                     # lattice constant (Angstrom)
        Eg=0.4, Band_offset=0.63,      # matches aestimo_database.py's InAs entry
        Ac=-5.08, Av=1.00,             # conduction/valence deformation potentials (eV)
        C11=83.29, C12=45.26, C44=39.59,  # elastic constants (GPa)
    ),
    'InSb': dict(
        m_e=0.0135, m_h=0.405,
        a0=6.4794,
        Eg=0.174, Band_offset=2.22,    # matches aestimo_database.py's InSb entry (derived value)
        Ac=-6.04, Av=0.31,
        C11=68.47, C12=37.35, C44=31.11,
    ),
}

for name, m in materials.items():
    m['fi_e'] = m['Band_offset'] * m['Eg']
    m['fi_h'] = -(1 - m['Band_offset']) * m['Eg']

matrix_mat, dot_mat = materials['InAs'], materials['InSb']
print(f"InAs (matrix): fi_e={matrix_mat['fi_e']:.4f} eV, fi_h={matrix_mat['fi_h']:.4f} eV")
print(f"InSb (dot):    fi_e={dot_mat['fi_e']:.4f} eV, fi_h={dot_mat['fi_h']:.4f} eV")

## Geometry: a spherical InSb dot in a box of InAs

In [ ]:
R_dot = 4.0    # quantum dot radius (nm)
L_half = 15.0  # half-width of the simulation box (nm) -- keep several dot radii of InAs padding
h = 1.0        # grid spacing (nm). Reduce this (and/or increase L_half) to check convergence.

coords = np.arange(-L_half, L_half + 1e-9, h)
N = len(coords)
X, Y, Z = np.meshgrid(coords, coords, coords, indexing='ij')
r = np.sqrt(X**2 + Y**2 + Z**2)
inside = r < R_dot

print(f"Grid: {N}^3 = {N**3:,} points, spacing {h} nm, box {2*L_half} nm, dot radius {R_dot} nm")
print(f"Points inside the dot: {inside.sum():,}")

## Strain: Eshelby's homogeneous-sphere solution

For a sphere of radius $R$ with a purely dilatational eigenstrain $\varepsilon^* \delta_{ij}$
embedded in an infinite, homogeneous, isotropic elastic matrix (Eshelby 1957), the resulting
strain is uniform and isotropic inside, and falls off as $1/r^3$ outside with **zero trace**
everywhere outside the sphere (the volume change is entirely accommodated inside):

$$\varepsilon_{ii}(r<R) = \varepsilon^* \cdot \frac{1+\nu}{3(1-\nu)} \quad\text{(each diagonal component, so Tr}\,\varepsilon = 3\times\text{this)}$$
$$\mathrm{Tr}\,\varepsilon(r>R) = 0$$

where $\nu$ is the (isotropic) Poisson ratio of the matrix. Since InAs is cubic, not isotropic,
$\nu$ is estimated here via a Voigt average of the elastic constants ($K$ = bulk modulus, $G$ =
shear modulus — $K$ is exact for cubic symmetry; $G$ is the approximation).

In [ ]:
def eigenstrain(a_dot, a_matrix):
    """Lattice-mismatch eigenstrain: how much the free-standing dot material must be
    strained to match the matrix lattice constant (negative = compressive)."""
    return (a_matrix - a_dot) / a_dot

def voigt_poisson_ratio(C11, C12, C44):
    K = (C11 + 2 * C12) / 3.0        # bulk modulus -- exact for cubic symmetry
    G = (C11 - C12 + 3 * C44) / 5.0  # shear modulus -- Voigt average (approximation)
    return (3 * K - 2 * G) / (2 * (3 * K + G))

eps_star = eigenstrain(dot_mat['a0'], matrix_mat['a0'])
nu = voigt_poisson_ratio(matrix_mat['C11'], matrix_mat['C12'], matrix_mat['C44'])
eps_inside_diag = eps_star * (1 + nu) / (3 * (1 - nu))
trace_inside = 3 * eps_inside_diag

print(f"Eigenstrain (lattice mismatch): {eps_star:.4f} ({eps_star*100:.2f}%)")
print(f"Matrix (InAs) Voigt Poisson ratio: {nu:.4f}")
print(f"Strain per axis inside dot: {eps_inside_diag:.4f}")
print(f"Tr(strain) inside dot: {trace_inside:.4f}, outside dot: 0 (by construction)")

trace_strain = np.where(inside, trace_inside, 0.0)

## Build the mass and potential fields

In [ ]:
m_e_field = np.where(inside, dot_mat['m_e'], matrix_mat['m_e'])
m_h_field = np.where(inside, dot_mat['m_h'], matrix_mat['m_h'])

# Unstrained band edge (piecewise) + hydrostatic strain shift (a_c/a_v * Tr(strain), nonzero
# only inside the dot per the Eshelby result above).
V_e = np.where(inside, dot_mat['fi_e'], matrix_mat['fi_e']) + dot_mat['Ac'] * trace_strain
V_h = np.where(inside, dot_mat['fi_h'], matrix_mat['fi_h']) + dot_mat['Av'] * trace_strain

print(f"Conduction band edge: InAs (matrix) = {matrix_mat['fi_e']:.3f} eV, "
      f"InSb (dot, strained) = {dot_mat['fi_e'] + dot_mat['Ac']*trace_inside:.3f} eV")
print(f"Valence band edge:    InAs (matrix) = {matrix_mat['fi_h']:.3f} eV, "
      f"InSb (dot, strained) = {dot_mat['fi_h'] + dot_mat['Av']*trace_inside:.3f} eV")
print()
print("Conduction band is LOWER in the matrix than in the (strained) dot => electrons are")
print("NOT confined by the dot. Valence band is HIGHER in the dot => holes ARE confined.")
print("This is the expected type-II (\"broken-gap\" family) behavior, consistent with the")
print("1D InAs/InSb/InAs notebook.")

### Sanity check: potential profile along a line through the center

In [ ]:
mid = N // 2
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(coords, V_e[:, mid, mid] * 1e3, color="#0072B2", label="Electron potential (Vc)")
ax.plot(coords, V_h[:, mid, mid] * 1e3, color="#E69F00", label="Hole potential (Vv)")
ax.axvspan(-R_dot, R_dot, color="0.85", zorder=0, label="InSb dot")
ax.set_xlabel("Position along x (nm)")
ax.set_ylabel("Energy (meV)")
ax.set_title("Band-edge cut through the dot center (before solving Schr\u00f6dinger)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

## Hamiltonian: 3D Ben Daniel-Duke finite differences, assembled sparse

$$H\psi = -\frac{\hbar^2}{2}\nabla\cdot\left(\frac{1}{m(\mathbf{r})}\nabla\psi\right) + V(\mathbf{r})\psi$$

discretized on the regular grid with the mass at each half-step evaluated as the harmonic mean
of the two neighboring grid points' masses (the standard prescription for probability-current
continuity across a mass discontinuity -- the same idea aestimo itself uses in 1D, e.g. in its
`Schro` function). Hard-wall (Dirichlet) boundaries are automatic: grid points simply aren't
connected past the array edge.

In [ ]:
def build_hamiltonian(mass_field, potential_field, h):
    """Assemble the sparse 3D Ben Daniel-Duke Hamiltonian (eV) on a cubic grid of spacing h (nm)."""
    Nx, Ny, Nz = mass_field.shape
    idx = np.arange(Nx * Ny * Nz).reshape(Nx, Ny, Nz)

    diag = potential_field.copy().astype(float)
    rows, cols, data = [], [], []

    for axis in range(3):
        m_next = np.take(mass_field, np.arange(1, mass_field.shape[axis]), axis=axis)
        m_here = np.take(mass_field, np.arange(0, mass_field.shape[axis] - 1), axis=axis)
        m_half = 2 * m_here * m_next / (m_here + m_next)  # harmonic mean
        t = HBAR2_OVER_2M0 / (m_half * h**2)              # coupling strength (eV)

        idx_here = np.take(idx, np.arange(0, idx.shape[axis] - 1), axis=axis)
        idx_next = np.take(idx, np.arange(1, idx.shape[axis]), axis=axis)

        rows.append(idx_here.ravel()); cols.append(idx_next.ravel()); data.append(-t.ravel())
        rows.append(idx_next.ravel()); cols.append(idx_here.ravel()); data.append(-t.ravel())

        diag_contrib = np.zeros_like(diag)
        slicer_here = [slice(None)] * 3; slicer_here[axis] = slice(0, -1)
        slicer_next = [slice(None)] * 3; slicer_next[axis] = slice(1, None)
        diag_contrib[tuple(slicer_here)] += t
        diag_contrib[tuple(slicer_next)] += t
        diag += diag_contrib

        # Hard-wall (Dirichlet) boundary: a ghost neighbor with psi=0 just outside the box
        # still contributes a diagonal term (just no off-diagonal coupling, since psi there
        # is fixed at zero). Without this, the two boundary faces along this axis would only
        # see their one real neighbor and behave like a reflecting (Neumann) wall instead.
        lo = [slice(None)] * 3; lo[axis] = 0
        hi = [slice(None)] * 3; hi[axis] = -1
        diag[tuple(lo)] += HBAR2_OVER_2M0 / (mass_field[tuple(lo)] * h**2)
        diag[tuple(hi)] += HBAR2_OVER_2M0 / (mass_field[tuple(hi)] * h**2)

    rows.append(idx.ravel()); cols.append(idx.ravel()); data.append(diag.ravel())

    rows = np.concatenate(rows); cols = np.concatenate(cols); data = np.concatenate(data)
    n = Nx * Ny * Nz
    H = sp.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
    return H

## Solve for confined states

Electrons solve the Hamiltonian directly. Holes are handled with the standard envelope-function
trick: a state confined where the valence band edge $V_h(\mathbf{r})$ is *highest* is found by
solving the Schr\u00f6dinger equation for $-V_h(\mathbf{r})$ (now an ordinary potential well) and
negating the resulting eigenvalues back — the same sign convention implicit in aestimo's own
hole-state energies (e.g. the negative hole-subband energies in the 1D notebooks).

In [ ]:
n_states = 4

H_e = build_hamiltonian(m_e_field, V_e, h)
sigma_e = V_e.min() - 1e-3
E_e, psi_e = spla.eigsh(H_e, k=n_states, sigma=sigma_e, which='LM')
order = np.argsort(E_e); E_e, psi_e = E_e[order], psi_e[:, order]

H_h_neg = build_hamiltonian(m_h_field, -V_h, h)
sigma_h_neg = (-V_h).min() - 1e-3
E_h_neg, psi_h = spla.eigsh(H_h_neg, k=n_states, sigma=sigma_h_neg, which='LM')
order = np.argsort(E_h_neg); E_h_neg, psi_h = E_h_neg[order], psi_h[:, order]
E_h = -E_h_neg  # flip back to the same absolute energy scale as V_h

cell_vol = h**3
psi_e = psi_e / np.sqrt((psi_e**2).sum(axis=0) * cell_vol)
psi_h = psi_h / np.sqrt((psi_h**2).sum(axis=0) * cell_vol)

print("Electron levels (meV):", np.round(E_e * 1e3, 2))
print("Hole levels (meV):    ", np.round(E_h * 1e3, 2))
print(f"\nNaive transition energy (E_e0 - E_h0): {(E_e[0]-E_h[0])*1e3:.1f} meV "
      f"({1239.84/(E_e[0]-E_h[0])/1000:.2f} um) -- see caveats above; ignores e-h Coulomb binding.")

# Sanity check: is the electron ground state basically just a particle-in-a-box state set by
# the simulation BOX SIZE rather than by the dot? (Expected in this type-II system.)
L_box = 2 * L_half
E_box_ground = 3 * HBAR2_OVER_2M0 * np.pi**2 / (matrix_mat['m_e'] * L_box**2)  # eV
print(f"\nFor comparison, a bare cubic box of side {L_box:.0f} nm (InAs mass, no dot) would give "
      f"a ground state of {E_box_ground*1e3:.1f} meV above the potential floor.")

## Visualize the confined states

In [ ]:
def line_cut_plot(coords, V_e, V_h, psi_e, psi_h, E_e, E_h, N, figsize=(9, 6)):
    mid = N // 2
    psi_e_grid = psi_e.reshape(N, N, N, -1)
    psi_h_grid = psi_h.reshape(N, N, N, -1)
    cuts_e = psi_e_grid[:, mid, mid, :]
    cuts_h = psi_h_grid[:, mid, mid, :]

    # Auto-scale so the tallest wavefunction cut spans roughly 40% of a typical level spacing.
    level_spacing = max(np.diff(np.sort(E_e)).mean(), np.diff(np.sort(E_h)).mean(), 1e-3) * 1e3
    amp_scale = 0.4 * level_spacing / max(np.abs(cuts_e).max(), np.abs(cuts_h).max())

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(coords, V_e[:, mid, mid] * 1e3, color="black", lw=1.5, label="Band edges")
    ax.plot(coords, V_h[:, mid, mid] * 1e3, color="black", lw=1.5)

    for n in range(cuts_e.shape[-1]):
        ax.plot(coords, cuts_e[:, n] * amp_scale + E_e[n] * 1e3, color="#0072B2", lw=1.0,
                label="Electron wavefunctions (x-cut)" if n == 0 else None)
        ax.axhline(E_e[n] * 1e3, color="#0072B2", lw=0.6, ls="--", xmin=0.05, xmax=0.95)
    for n in range(cuts_h.shape[-1]):
        ax.plot(coords, cuts_h[:, n] * amp_scale + E_h[n] * 1e3, color="#E69F00", lw=1.0,
                label="Hole wavefunctions (x-cut)" if n == 0 else None)
        ax.axhline(E_h[n] * 1e3, color="#E69F00", lw=0.6, ls="--", xmin=0.05, xmax=0.95)

    ax.axvspan(-R_dot, R_dot, color="0.9", zorder=0)
    ax.set_xlabel("Position along x, through dot center (nm)")
    ax.set_ylabel("Energy (meV)")
    ax.set_title("Line cut through the dot center")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    return fig

fig1 = line_cut_plot(coords, V_e, V_h, psi_e, psi_h, E_e, E_h, N)

In [ ]:
def slice_plot(coords, psi_h, N, state=0, figsize=(6, 5)):
    mid = N // 2
    psi_h_grid = psi_h.reshape(N, N, N, -1)
    density = psi_h_grid[:, :, mid, state]**2

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.pcolormesh(coords, coords, density.T, shading='auto', cmap='viridis')
    circle = plt.Circle((0, 0), R_dot, fill=False, color='white', ls='--', lw=1.2)
    ax.add_patch(circle)
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
    ax.set_title(f"Hole ground state $|\\psi|^2$, z=0 slice")
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, label="Probability density (nm$^{-3}$)")
    fig.tight_layout()
    return fig

fig2 = slice_plot(coords, psi_h, N, state=0)